In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.environ["GEMINI_API_KEY"]

print("working")

working


In [2]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])

0

In [3]:
import sys
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "gitsource"])

0

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [5]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

# Question 1

In [6]:
print(len(documents))

72


# Question 2

In [7]:
import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

############################################

result = index.search(
    query="How does the agentic loop keep calling the model until it stops?"
)

print(result[0]["filename"])

01-agentic-rag/lessons/14-agentic-loop.md


In [8]:
import minsearch
print(dir(minsearch))

['AppendableIndex', 'DEFAULT_ENGLISH_STOP_WORDS', 'Highlighter', 'Index', 'STEMMERS', 'Tokenizer', 'VectorSearch', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'append', 'filters', 'get_stemmer', 'highlighter', 'lancaster_stemmer', 'minsearch', 'porter_stemmer', 'snowball_stemmer', 'stemmers', 'tokenizer', 'vector']


# Question 3

In [9]:
import requests
r = requests.get("https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py")
with open("rag_helper.py", "w") as f:
    f.write(r.text)
print("Saved to:", os.getcwd())

Saved to: e:\VS Code stuff\LLMs zoomcamp


In [10]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "google-generativeai"])

0

In [19]:
import google.generativeai as genai

from importlib import reload
import rag_helper
reload(rag_helper)

from rag_helper import RAGBase

genai.configure(api_key=os.environ.get("GEMINI_API_KEY"))
genai_client = genai.GenerativeModel("gemini-2.5-flash")

rag = RAGBase(index=index, llm_client=genai_client)

answer, input_tokens = rag.rag(
    "How does the agentic loop keep calling the model until it stops?"
)

print("Answer:", answer)
print("Input tokens:", input_tokens)

Answer: The agentic loop keeps calling the model until it stops by using a `while` loop that continues as long as the model's response contains function calls.

Here's a breakdown of how it works:

1.  **Initial Call:** The process starts by sending the user's question and agent instructions to the model.
2.  **Process Response:** The agent code receives the model's response.
    *   If the response includes a `function_call` (e.g., to `search`), the agent executes that function.
    *   The results of the function call are then appended to the conversation history (`messages`).
    *   A flag (e.g., `has_function_calls`) is set to `True` if any function calls were made.
    *   If the response is a `message` (a direct answer), it's printed, and the `has_function_calls` flag remains `False` (or is reset if it was previously `True`).
3.  **Loop Condition:** The `while` loop continues to iterate. After processing a response, it checks the `has_function_calls` flag.
    *   **Keep Looping

# Question 4

In [20]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(len(chunks))

295


# Question 5

In [21]:
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks) 

In [22]:
rag_chunks = RAGBase(index=chunk_index, llm_client=genai_client)

answer, input_tokens = rag_chunks.rag(
    "How does the agentic loop keep calling the model until it stops?"
)

print("Input tokens (chunked):", input_tokens)
print("Input tokens (original):", 7947)
print("Ratio:", 7947 / input_tokens)

Input tokens (chunked): 2599
Input tokens (original): 7947
Ratio: 3.0577145055790687


# Question 6

In [23]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "toyaikit"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "openai"])

print("DONE")

DONE


In [29]:

'''
it looks like toyaikit doesn't work with gemini, so i will write the agentic loop myself
'''
import json
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.environ.get("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

def search(query: str) -> list:
    """Search the course materials for content matching the given query."""
    return chunk_index.search(query, num_results=5)

tools = [{
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the course materials for content matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The search query"}
            },
            "required": ["query"]
        }
    }
}]

instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": "How does the agentic loop work, and how is it different from plain RAG?"}
]

search_count = 0

while True:
    response = client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=messages,
        tools=tools
    )
    
    msg = response.choices[0].message
    messages.append(msg)
    
    if msg.tool_calls:
        for tool_call in msg.tool_calls:
            search_count += 1
            args = json.loads(tool_call.function.arguments)
            print(f"Searching: {args['query']}")
            result = search(args["query"])
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })
    else:
        print("\nFinal answer:", msg.content)
        break

print(f"\nSearch was called {search_count} times")

Searching: agentic loop
Searching: RAG vs agentic loop

Final answer: The agentic loop and plain RAG both involve Large Language Models (LLMs) and information retrieval, but they differ significantly in their approach to problem-solving and control flow.

Here's a breakdown:

### Agentic Loop

The agentic loop is a foundational pattern for AI agents, characterized by an iterative and dynamic process. Instead of a fixed sequence, the LLM acts as a central decision-maker within a loop:

1.  **LLM Call:** The process begins with an LLM call based on a user prompt or current goal.
2.  **Tool Execution (Optional):** The LLM may decide it needs more information or to perform an action. It will then invoke a tool (e.g., search, code execution, API call) and pass the necessary arguments.
3.  **Result Feedback:** The results from the tool execution are then fed back to the LLM.
4.  **Iteration/Refinement:** Based on these new results, the LLM re-evaluates its state. It can decide to:
    *   Ca